## AI for Social Good Prototype — SDG 2: Zero Hunger

**Topic:** End hunger, achieve food security and improved nutrition, and promote sustainable agriculture.

This Colab notebook is structured around the project requirements. It adapts Lab 1, Lab 2, and Lab 3 for a local Zero Hunger system.

## Section 1: Problem and Population

Diego is a first-year SJSU student who works evenings, lives off campus, and sometimes skips meals because he does not know which food pantry or community food resource can help him today. The failure point is not that food resources do not exist; it is that information is scattered, pantry intake questions are hard to answer quickly, and dietary needs such as halal, vegetarian, allergies, or low-sodium meals can be missed during intake. This project supports **SDG 2: Zero Hunger** by helping food-insecure students and nearby residents get clearer next steps while helping pantry staff turn messy requests and donation information into usable records. Since M1, the topic stayed focused on Zero Hunger, but the scope is narrowed from the broad goal of “ending hunger” to a specific local workflow: food help intake, resource matching, and donation triage, because a narrower problem can be tested in a working prototype.

## Section 2: Proposed System

**System name:** FoodLink AI Intake and Donation Assistant

**Structured workflow:**

Student send request or donation photo -> Google Gemini processes generation by structured extraction + visual recognition -> food need record and donation assessment -> Language output -> pantry and basic needs staff review generation result  -> real-world action

| Steps | What happens |
|---|---|
| **1. Input** | A student or resident submits a food help message, or a volunteer uploads a photo of donated food. |
| **2. AI processing** | Lab 1 drafts a respectful reply in written language. Lab 2 extracts structured fields like urgency, dietary needs, transportation barriers, and recommended action. Lab 3 analyzes a donation photo for visible item type, condition, and whether a human should inspect it. |
| **3. Output** | The system produces a response, a pantry intake record, and a donation assessment. |
| **4. Real-world action** | A food pantry worker or basic needs staff member reviews the AI output, contacts the person, confirms availability, then routes them to food pickup, emergency support, or donation handling. |

## Section 3: Project Code

This section modifies **Lab 1: Text Generation**, **Lab 2: Structured Data Extraction**, and **Lab 3: Visual Recognition** for the Zero Hunger project. Lab 1 supports communication with the person asking for help. Lab 2 turns a messy food-help request into fields pantry staff can use. Lab 3 supports donation intake by analyzing a photo of a donated food item or pantry shelf.

In [1]:

!pip -q install google-genai pydantic

from google import genai
from google.genai import types
from getpass import getpass
from pydantic import BaseModel, Field
from typing import Literal
import json
import textwrap

API_KEY = getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

MODEL = "gemini-2.5-flash"
print("Gemini client configured successfully. Model:", MODEL)

Paste your Gemini API key: ··········
Gemini client configured successfully. Model: gemini-2.5-flash


### Lab 1 Reference: Food Resource Response Generator

 This lab shows plain language food support message for a student or resident. It avoids promising that food is available until staff confirm it.

In [2]:

def draft_food_resource_reply(request_text):
    system_prompt = """
You are FoodLink AI, an assistant for a campus/community food pantry intake team.
Your job is to draft a short, respectful reply to a person asking for food support.

Rules:
- Detect the user's language and respond in that same language when possible.
- Use simple, supportive language.
- Do not promise that food, appointments, or delivery are available until staff confirm it.
- Ask for only the missing information needed for intake.
- If the person says they have not eaten today, have no safe food, or need food tonight, mark the situation as urgent and tell them staff should review it the same day.
- If there is immediate medical danger, tell the person to contact emergency services.
- Mention that a pantry/basic-needs staff member should verify resources before action is taken.
"""

    user_prompt = f"""
Food help request:
{request_text}

Draft the response in 5 sentences or fewer.
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=0.2
        )
    )
    return response.text

sample_request = """ I am an SJSU student and I have not eaten since yesterday. I do not have a car.
I need halal food near campus today and I only have time before my evening shift. """

reply = draft_food_resource_reply(sample_request)
print(reply)

Thank you for reaching out. We understand you need halal food today and haven't eaten since yesterday, which makes this urgent. To help us check availability for you, please provide your full name and student ID. A pantry staff member will review your request as soon as possible to see what support we can offer before your evening shift. Please know that staff should review this request today.


**What this demonstrates:** Using Lab 1's as a reference, this shows how the system can communicate with a food needy student in a supportive way, identifying urgency, and asking for missing intake details without promising that resources are guaranteed. This matters because the first response can either reduce confusion or make the person wait longer for help.

### Lab 2 Reference: Structured Food Intake Extraction

This lab turns a messy food help request into a structured intake record that pantry staff can look through quickly.

In [3]:

class FoodIntakeRecord(BaseModel):
    detected_language: str = Field(description="Language or languages used by the requester")
    location: str = Field(description="Location mentioned by the requester, or 'unknown'")
    need_type: Literal[
        "immediate_food",
        "ongoing_food_support",
        "nutrition_guidance",
        "donation_question",
        "other"
    ] = Field(description="Main category of the request")
    urgency: Literal["low", "medium", "high", "emergency"] = Field(description="Urgency level")
    household_size: int | None = Field(description="Number of people needing food, if mentioned")
    dietary_needs: list[str] = Field(description="Dietary restrictions, allergies, nutrition needs, or preferences")
    transportation_barrier: bool = Field(description="Whether the person has trouble getting to resources")
    time_constraint: str = Field(description="Any deadline or limited availability mentioned")
    recommended_action: str = Field(description="Recommended next step for pantry/basic-needs staff")
    human_review_required: bool = Field(description="True if staff should review before action is taken")
    reason_for_review: str = Field(description="Why human review is needed, or 'none'")


def extract_food_intake(request_text):
    prompt = f"""
Extract a structured intake record from this food-help request.
Use the schema exactly. Do not invent information that is not present.
If the person says they have not eaten today, need food tonight, has medical risk, has a baby/senior involved,
or has unclear dietary restrictions, set human_review_required to true.

Food-help request:
{request_text}
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=FoodIntakeRecord,
            temperature=0.1
        )
    )
    return response.text

intake_json = extract_food_intake(sample_request)
print(intake_json)

{
  "detected_language": "English",
  "location": "SJSU campus area",
  "need_type": "immediate_food",
  "urgency": "emergency",
  "household_size": null,
  "dietary_needs": ["halal"],
  "transportation_barrier": true,
  "time_constraint": "before evening shift",
  "recommended_action": "Locate immediate halal food resources near SJSU campus accessible before evening shift, considering no car.",
  "human_review_required": true,
  "reason_for_review": "Requester has not eaten since yesterday, indicating high immediate need."
}


**What this demonstrates:** This Lab 2 modification shows how the system converts a free text food request into consistent fields such as urgency, dietary needs, transportation barrier, and human review status. This supports the proposed system because staff need structured information before they can decide who needs same day help and what food restrictions must be respected.

### Lab 3 Modified: Donation Image Recognition

This lab asks to upload a clear photo of a pantry shelf, canned food, boxed food, or donated item when the cell asks for a file.

In [5]:

from google.colab import files

class DonationImageAssessment(BaseModel):
    visible_items: list[str] = Field(description="Food items visibly present in the image")
    estimated_condition: Literal[
        "appears_good",
        "check_expiration",
        "damaged_packaging",
        "unsafe_or_unclear"
    ] = Field(description="Visual condition estimate")
    nutrition_category: list[Literal[
        "produce", "grain", "protein", "dairy", "snack", "beverage", "mixed", "other"
    ]] = Field(description="Broad nutrition or food categories visible")
    possible_allergen_or_diet_notes: list[str] = Field(description="Visible allergen or dietary notes, if any")
    confidence_level: Literal["low", "medium", "high"] = Field(description="Confidence based only on the image")
    recommended_action: str = Field(description="What pantry staff should do next")
    human_review_required: bool = Field(description="Whether staff must inspect the item before distributing")

print("Upload a photo of a donated food item, pantry shelf, or food package.")
uploaded = files.upload()
image_path = next(iter(uploaded))

image_file = client.files.upload(file=image_path)

vision_prompt = """
You are helping a food pantry triage donated food from an image.
Analyze only what is visible in the image.
Do not guess expiration dates if they are not visible.
If packaging is damaged, labels are unreadable, the item is homemade, or the safety status is unclear,
set human_review_required to true.
Return the structured assessment using the schema.
"""

vision_response = client.models.generate_content(
    model=MODEL,
    contents=[image_file, vision_prompt],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=DonationImageAssessment,
        temperature=0.1
    )
)

print(vision_response.text)

Upload a photo of a donated food item, pantry shelf, or food package.


Saving hamhelper.htm to hamhelper.htm
{
  "visible_items": ["Hamburger Helper Cheesy Enchilada"],
  "estimated_condition": "appears_good",
  "nutrition_category": ["grain", "mixed"],
  "possible_allergen_or_diet_notes": [],
  "confidence_level": "high",
  "recommended_action": "Stock item for distribution.",
  "human_review_required": false
}


**What this demonstrates:** This Lab 3 modification shows how image recognition can support food donation intake by identifying visible items, flagging damaged or unclear packaging, and letting staff know when a review is needed. This will not replace a pantry worker; it gives them a tool so they can sort donations faster and more safely.

## Section 4: Edge Case Elicitation

This section intentionally tries to surface a failure. The edge case targets a user outside the assumed majority: a mixed-language requester with a senior family member, no transportation, urgent food need, and a vague nutrition restriction. It also includes a safety issue with a donated item that may make the model's confidence misleading.

In [6]:

edge_case_prompt = """
Hola, necesito comida para hoy. My grandma is 78 and diabetic, but I don't know what food is safe for her.
We have no car and I can only text because my phone plan has limited minutes.
A neighbor gave us a dented can with no label, and I am not sure if we should eat it tonight.
There are 3 people in my home.
"""

print("EDGE CASE PROMPT:")
print(edge_case_prompt)
print("\n--- LAB 1 RESPONSE ---")
print(draft_food_resource_reply(edge_case_prompt))
print("\n--- LAB 2 STRUCTURED EXTRACTION ---")
print(extract_food_intake(edge_case_prompt))


EDGE CASE PROMPT:

Hola, necesito comida para hoy. My grandma is 78 and diabetic, but I don't know what food is safe for her.
We have no car and I can only text because my phone plan has limited minutes.
A neighbor gave us a dented can with no label, and I am not sure if we should eat it tonight.
There are 3 people in my home.


--- LAB 1 RESPONSE ---
Hola. Entendemos que necesitas comida hoy y estamos aquí para ayudarte. Por favor, no consuman la lata abollada sin etiqueta; es mejor evitar riesgos. Un miembro de nuestro equipo de FoodLink se comunicará contigo lo antes posible para discutir opciones, incluyendo alimentos seguros para tu abuela y posible entrega. Para que podamos coordinar mejor, ¿podrías compartir tu dirección o ubicación general? Esta situación es urgente y el personal la revisará hoy mismo.

--- LAB 2 STRUCTURED EXTRACTION ---
{"detected_language": "English, Spanish", "location": "unknown", "need_type": "immediate_food", "urgency": "emergency", "household_size": 3, 

**Prompt used:** The edge case prompt describes a mixed Spanish/English request from a household with a 78-year-old diabetic grandmother, no car, limited phone access, urgent food need, and a dented unlabeled can.

**Output returned:** After running the code cell above, leave the Lab 1 response and Lab 2 output visible here in the notebook.

**One-sentence assessment:** This is a close miss if Gemini flags urgent human review but gives only general food advice. It becomes a failure if it suggests eating the dented unlabeled can, ignores the diabetes issue, or fails to mark the case for human review.

## Section 5: Oversight & Tradeoffs

FoodLink AI's use of human review is a non-negotiable requirement. It is needed for verifiation of resource availability, ensuring the safety of food that are received, and the overall inspection of the donations. Due to the high use of human review though trade offs begin to arise. The more manual review that is used causes a slower process of resolution for the donator or requestee. Although a slower process, safety will always be our number one priority.